# EJERCICIO METRICAS DE CLASIFICACION

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score, precision_score, precision_recall_curve
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix


import warnings

warnings.filterwarnings('ignore')


%matplotlib inline



In [2]:
def categoricas_unicos(dataframe, var_cat, vc_out=True):
    """
    Muestra valores unicos y conteo de las variables categóricas.
    -------------------------------
    Parametros de entrada:
    - dataframe: DF de pandas
    - var_cat: lista de variables categóricas (cadena)
    - vc_out: booleano para mostrar o no (True/False) el resultado de "value_counts()"
        Por defecto lo muestra

    -------------------------------
    SALIDA:
    No devuelve valor. Saca por pantalla la información

    """

    for discreta in var_cat:

        print(f'Variable {discreta.upper()}:')
        print('Valores unicos: ')
        print(dataframe[discreta].unique(), end='\n'*2)

        if vc_out:
            print(dataframe[discreta].value_counts(), end='\n'*2)





In [3]:
def graficas_var_categorica(dataframe, var_cat):
    """
    Realiza los diagramas de barras de las variables categóricas de un dataframe
    -------------------------------
    Parametros de entrada:
    - dataframe: DF de pandas
    - var_cat: lista de variables numéricas (cadena)

    -------------------------------
    SALIDA:
    No devuelve valor. Dibuja las gráficas por pantalla

    """


    colores = sns.color_palette("husl", len(var_cat))

    # creacion matriz de graficas
    fig, axes = plt.subplots(len(var_cat), 1, \
                             figsize=(10, 4*len(var_cat)),\
                             gridspec_kw={'hspace': 0.4, 'wspace': 0.4})

    ax = axes.ravel()

    # dibujamos las graficas
    for idx,variable in enumerate(var_cat):

        sns.countplot(dataframe[variable], ax=ax[idx],palette=colores)

        ax[idx].set_title(f'DIAGRAMA DE BARRAS {variable}')
        ax[idx].set_xlabel(f'Valores únicos (categorias) de {variable}')
        ax[idx].set_ylabel("Frequencia")


In [4]:
def graf_histo_box_numericas(dataframe, var_num):
    """
    Realiza graficas histograma y boxplot de variables numéricas de un dataframe
    -------------------------------
    Parametros de entrada:
    - dataframe: DF de pandas
    - var_num: lista de variables numéricas (cadena)

    -------------------------------
    SALIDA:
    No devuelve valor. Dibuja las gráficas por pantalla

    """

    TABLEAU_CMP = ('tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', \
                   'tab:gray','tab:olive', 'tab:cyan')

    fig, axes = plt.subplots(len(var_num), 2, \
                             figsize=(20, 5 * len(var_num)), \
                             gridspec_kw={'hspace': 0.4, 'wspace': 0.1})
    ax = axes.ravel()

    # graficas distribucion y boxplot de cada atributo
    for idx, atributo in enumerate(var_num):

        # distribucion (histograma)
        sns.distplot(dataframe[atributo], bins=30, ax=ax[2 * idx], \
                     color=TABLEAU_CMP[idx % len(TABLEAU_CMP)], \
                     hist_kws={'alpha': 0.15})

        # titulo, etiquetas histograma
        ax[2 * idx].set_title(f'HISTOGRAMA {atributo}')
        ax[2 * idx].set_xlabel(f'Valores {atributo}')
        ax[2 * idx].set_ylabel("Frequencia")

        # boxplot
        sns.boxplot(x=atributo, data=dataframe, ax=ax[2 * idx + 1], color=TABLEAU_CMP[idx % len(TABLEAU_CMP)])

        # titulo, etiquetas boxplot
        ax[2 * idx + 1].set_title(f'BOXPLOT {atributo}')
        ax[2 * idx + 1].set_xlabel(f'Valores {atributo}')



## DATASET DIABETES

Vamos a utilizar este dataset para este ejercicio de métricas de clasificación. Realizamos las primeras correcciones:

In [ ]:
RUTA = '/content/drive/MyDrive/ESPECIALISTA IA/diabetes.csv'

df_diabetes = pd.read_csv(RUTA)

df_diabetes.info()

In [ ]:
df_diabetes.columns

In [ ]:
col_pasar_numero = ['chol_hdl_ratio', 'bmi', 'waist_hip_ratio']

In [ ]:
categoricas_unicos(df_diabetes,col_pasar_numero)

### PROBLEMA COMÚN EN LIMPIEZA: DATOS NUMÉRICOS COMO CADENA Y CON LA COMA COMO SEPARADOR DE DECIMALES

In [ ]:

for col in col_pasar_numero:

    df_diabetes[col] = df_diabetes[col].str.replace(",",".").astype('float')




In [ ]:
df_diabetes.info()

**Eliminamos campo ID:**

In [ ]:
df_diabetes.drop(columns='patient_number', inplace=True)

In [ ]:


df_diabetes.columns

In [ ]:
df_diabetes.diabetes.value_counts()

**Separamos atributos de target y tratamos éste último:**

In [ ]:
target = 'diabetes'

X = df_diabetes.drop(columns=target)

y = df_diabetes[target]



In [ ]:
from sklearn.preprocessing import LabelEncoder

lbl_encoder = LabelEncoder()

y_num = lbl_encoder.fit_transform(y)

y_num

In [ ]:
y

Codificacion TARGET: (0) DIABETES; (1) NO DIABETES

**Separamos por tipo y echamos un vistazo a los atributos:**

In [ ]:
var_num = X.select_dtypes(exclude='object').columns.to_list()

var_cat = X.select_dtypes(include='object').columns.to_list()

var_cat

In [ ]:
var_num

In [ ]:
graf_histo_box_numericas(df_diabetes, var_num)

In [ ]:
sns.countplot(data=df_diabetes, x='gender')

## PARTICIÓN TRAIN/TEST. TRANSFORMADOR "COLUMN TRANSFORMER"

In [ ]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y_num, \
                                                test_size=0.2, \
                                                stratify=y_num, random_state=42)

Xtrain.shape, Xtest.shape, ytrain.shape, ytest.shape

In [ ]:
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler, RobustScaler

from sklearn.preprocessing import OneHotEncoder

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

from sklearn.naive_bayes import GaussianNB

**Primero instanciamos, depués transformamos de la manera adecuada (train/test):**

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('scale', RobustScaler(), var_num),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), var_cat)],
    remainder='passthrough')

In [ ]:
preprocessor

In [ ]:
preprocessor.fit(Xtrain)

Xtrain_prx = preprocessor.transform(Xtrain)

Xtest_prx = preprocessor.transform(Xtest)


## Modelo base KNN

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(Xtrain_prx, ytrain)

knn.score(Xtrain_prx, ytrain)

In [ ]:
knn.score(Xtest_prx, ytest)

0: si diabetes
1: no diabetes


#### Nos vamos a ocupar de los FP

In [ ]:
y_predict_knn = knn.predict(Xtest_prx)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score
from sklearn.metrics import confusion_matrix

mat_conf = confusion_matrix(ytest, y_predict_knn)

mat_conf



In [ ]:
# crear dataframe auxiliar
dataframe = pd.DataFrame(mat_conf, index=['maligno', 'benigno'], columns=['maligno', 'benigno'])

# crear heatmap
sns.heatmap(dataframe, annot=True, cbar=True, cmap='Reds')
plt.title('Confusion Matrix')

plt.tight_layout(), plt.xlabel('Predicted Values'), plt.ylabel('True Values');


Veamos la precisión:

In [ ]:
from sklearn.metrics import precision_score

precision_score(ytest, y_predict_knn)

**Valor bastante alto**

## Modelo Naive Bayes

Entrenamos el modelo:

In [ ]:
nb = GaussianNB().fit(Xtrain_prx,ytrain)

nb.score(Xtest_prx, ytest)

In [ ]:
y_pred_nb = nb.predict(Xtest_prx)

Veamos las métricas y la matriz de confusión:

In [ ]:
accuracy_score(ytest, y_pred_nb)

In [ ]:
precision_score(ytest, y_pred_nb)

In [ ]:
mat_conf = confusion_matrix(ytest, y_pred_nb)

# crear dataframe auxiliar
dataframe = pd.DataFrame(mat_conf, index=['diabetes', 'no_diabetes'], columns=['diabetes', 'no_diabetes'])

# crear heatmap
sns.heatmap(dataframe, annot=True, cbar=True, cmap='Reds')
plt.title('Confusion Matrix')

plt.tight_layout(), plt.xlabel('Predicted Values'), plt.ylabel('True Values')

**MIsmo número de falsos positivos (precision), score un poco mas bajo que el KNN**

Si quisieramos un **informe general con todas las métricas** utilizaremos **"classification_report"** (para binaria fijarse solo en la fila de la clase positiva):


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(ytest, y_pred_nb))

### CURVAS PRECISION-RECALL

In [ ]:
y_pred_proba_nb = nb.predict_proba(Xtest_prx)[:,1]

**Pasamos a dibujar la curva con la función "precision_recall_curve" pasándole como parámetros "y_test" e "y_pred_proba":**

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(ytest, y_pred_proba_nb)

plt.fill_between(recall, precision)

plt.ylabel("Precision")
plt.xlabel("Recall")
plt.title("Train Precision-Recall curve");

### CURVAS ROC. Area bajo la curva AUC

Las **curvas ROC** se obtienen procediendo de forma similar, esta vez con la función **"roc_curve":**

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(ytest, y_pred_proba_nb)

plt.plot(fpr, tpr)

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

Y podemos conocer el área bajo la curva con **"roc_auc_score"**, que **nos da una buena medida general de lo bueno que es nuestro clasificador:**

In [ ]:
from sklearn.metrics import roc_auc_score

roc_auc_score(ytest, y_pred_proba_nb)

## COMPARACION DE LOS 2 MODELOS

Finalmente podemos **comparar los dos modelos, el base KNN y el utilizado NB**. Calculamos primero la predicción en probabilidad del KNN:

In [ ]:
y_pred_proba_knn = knn.predict_proba(Xtest_prx)[:,1]

fpr_knn, tpr_knn, threshold_knn = roc_curve(ytest, y_pred_proba_knn)

In [ ]:
# Dibujamos con todos los datos calculados

plt.plot(fpr, tpr, marker='^', label='Naive Bayes')
plt.plot(fpr_knn, tpr_knn, marker='.', label='KNN')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

plt.legend()

In [ ]:
print(f'AUC (KNN): {roc_auc_score(ytest, y_pred_proba_knn)}')

print(f'AUC (Naive Bayes): {roc_auc_score(ytest, y_pred_proba_nb)}')



**CONCLUSIÓN: con similar "precision", las curvas y áre ROC nos dicen que el Naive Bayes se comporta ligeramante mejor**